In [1]:
import pandas as pd 
import numpy as np
from dotenv import load_dotenv
import csv
import os
import yaml
load_dotenv()
from config.settings import PROCESSED_DATA_PATH, yaml_config_path

In [ ]:
with open(yaml_config_path, "r") as f:
    config = yaml.safe_load(f)

categories = config.get ("categories", [])

Target_categories = "Baby"

if Target_categories not in categories:
    raise ValueError(f"Target category '{Target_categories}' not found in YAML config")

print(f"Processing category: {Target_categories}")

path = os.path.join(PROCESSED_DATA_PATH, "raw", "amazon_reviews")
category_path = os.path.join(path, f"category={Target_categories}")

parquet_file_path = [
    os.path.join(category_path, f)
    for f in os.listdir(category_path)
    if f.endswith(".parquet")
]

if not parquet_file_path:
    raise FileNotFoundError(f"No parquet files found in {category_path}")

print(f"Found parquet files: {parquet_file_path}")

dfs = []

for file in parquet_file_path[:5]:  # Limit to first 5 files for testing
    try:
        df = pd.read_parquet(file)
        dfs.append(df)
        print(f"Successfully read {file}")
    except Exception as e:
        print(f"Error reading {file}: {e}")

# ===== COMBINE =====
final_df = pd.concat(dfs, ignore_index=True)

print(f"Total rows: {len(final_df)}")

# ===== SAVE OUTPUT =====
output_file = os.path.join(path, "part_00001.parquet")
final_df.to_parquet(output_file, index=False)

print(f"Saved to: {output_file}")

In [6]:
final_df.head()


,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,9970739,R8EWA1OFT84NX,B00GSP5D94,329991347,Summer Infant SwaddleMe Adjustable Infant Wrap...,Baby,5,0,0,False,True,Great swaddled blankets,Loved these swaddle blankets and so did my dau...,2015-08-31
1,US,23538442,R2JWY4YRQD4FOP,B00YYDDZGU,646108902,Pacifier Clip Girl (3 Pack) Ziggy Baby 2-Sided...,Baby,5,0,0,False,False,Too cute and really nice,These are adorable pacifier clips. SavvyBaby h...,2015-08-31
2,US,8273344,RL5ESX231LZ0B,B00BUBNZC8,642922361,Udder Covers - Breast Feeding Nursing Cover,Baby,5,0,0,False,True,Five Stars,Great gift,2015-08-31
3,US,24557753,RRMS9ZWJ2KD08,B00AWLZFTS,494272733,Gerber Graduates Fun Pack Utensils,Baby,5,0,0,False,True,Cute; wash up nicely in dishwasher.,These forks are great for my 10 month old daug...,2015-08-31
4,US,46263340,R14I3ZG5E6S7YM,B00KM60D3Q,305813185,Summer Infant Ultra Sight Pan/Scan/Zoom Video ...,Baby,5,0,0,False,True,Love it!,I wanted something for piece of mind with my l...,2015-08-31


In [7]:
print(final_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   marketplace        1000000 non-null  category      
 1   customer_id        1000000 non-null  int64         
 2   review_id          1000000 non-null  object        
 3   product_id         1000000 non-null  object        
 4   product_parent     1000000 non-null  int64         
 5   product_title      1000000 non-null  object        
 6   product_category   1000000 non-null  category      
 7   star_rating        1000000 non-null  int8          
 8   helpful_votes      1000000 non-null  int16         
 9   total_votes        1000000 non-null  int16         
 10  vine               1000000 non-null  bool          
 11  verified_purchase  1000000 non-null  bool          
 12  review_headline    999995 non-null   object        
 13  review_body        999842 no

In [8]:
path = "E:/e-commerce-dataset/archive"

# Load the dataset
df = pd.read_csv(f"{path}/amazon_reviews_multilingual_US_v1_00.tsv", sep='\t')

df.head()

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
import pandas as pd
import csv

path = "E:/e-commerce-dataset/archive/amazon_reviews_multilingual_US_v1_00.tsv"

total_rows = 0
bad_rows = 0

def bad_line_handler(line):
    global bad_rows
    bad_rows += 1
    return None

reader = pd.read_csv(
    path,
    sep="\t",
    chunksize=100_000,
    on_bad_lines=bad_line_handler,
    engine="python",
    quoting=csv.QUOTE_NONE
)

for i, chunk in enumerate(reader):
    total_rows += len(chunk)
    print(f"Chunk {i} processed, total_rows={total_rows}")

print("FINAL:")
print("Total valid rows:", total_rows)
print("Bad rows:", bad_rows)

Chunk 0 processed, total_rows=100000
Chunk 1 processed, total_rows=200000
Chunk 2 processed, total_rows=300000
Chunk 3 processed, total_rows=400000
Chunk 4 processed, total_rows=500000
Chunk 5 processed, total_rows=600000
Chunk 6 processed, total_rows=700000
Chunk 7 processed, total_rows=800000
Chunk 8 processed, total_rows=900000
Chunk 9 processed, total_rows=1000000
Chunk 10 processed, total_rows=1100000
Chunk 11 processed, total_rows=1200000
Chunk 12 processed, total_rows=1300000
Chunk 13 processed, total_rows=1400000
Chunk 14 processed, total_rows=1500000
Chunk 15 processed, total_rows=1600000
Chunk 16 processed, total_rows=1700000
Chunk 17 processed, total_rows=1800000
Chunk 18 processed, total_rows=1900000
Chunk 19 processed, total_rows=2000000
Chunk 20 processed, total_rows=2100000
Chunk 21 processed, total_rows=2200000
Chunk 22 processed, total_rows=2300000
Chunk 23 processed, total_rows=2400000
Chunk 24 processed, total_rows=2500000
Chunk 25 processed, total_rows=2600000
Chunk

In [ ]:
import pyarrow.csv as pv
import pyarrow.parquet as pq
import pyarrow as pa
import os

path = "E:/e-commerce-dataset/archive/amazon_reviews_multilingual_US_v1_00.tsv"
output_dir = "E:/ecommerce-data-lake/raw/category=multilingual"

os.makedirs(output_dir, exist_ok=True)

with pv.open_csv(
    path,
    read_options=pv.ReadOptions(block_size=20 * 1024 * 1024),
    parse_options=pv.ParseOptions(
        delimiter="\t",
        invalid_row_handler=lambda row: "skip"   # 🔥 FIX
    )
) as reader:

    for i, batch in enumerate(reader):
        table = pa.Table.from_batches([batch])   # 🔥 FIX

        pq.write_table(
            table,
            f"{output_dir}/part_{i}.parquet"
        )

        print(f"Saved part_{i}")

In [ ]:
df = pd.read_parquet(f"{output_dir}/part_170.parquet")
df.head(200)

,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,41897336,RESZS1DT428IL,B005544TRQ,436223062,Suits Season 1,Digital_Video_Download,5,0,0,N,Y,"Great series! Very mature, but silly at times....",So glad my brother and sister in law told me a...,2015-07-22
1,US,2334153,R135H36N1YGMW,B007B5EDEG,268067011,A Game of Thrones: The Story Continues Books 1...,Digital_Ebook_Purchase,5,0,1,N,Y,Five Stars,I love you,2015-07-22
2,US,41760232,R3KA16WE35LJB9,B004ZMWUCU,52387917,The Misremembered Man,Digital_Ebook_Purchase,5,0,0,N,Y,Very believable and engrossing.,"As I was reading this book, I was thinking how...",2015-07-22
3,US,2008108,R4QID2938NTKX,B00CTKHXFO,101368587,Metal Gear Solid: The Legacy Collection,Video Games,5,2,3,N,Y,WARNING WHEN BUYING FROM AMAZON!,WARNING: Be careful when ordering this collect...,2015-07-22
4,US,41755705,RK2B9X4IS85NS,B00HZ3C4N6,263154361,Jack Ryan: Shadow Recruit,Digital_Video_Download,3,0,0,N,Y,Three Stars,Was OK - a little slow and kind of hard to fol...,2015-07-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,US,19916590,R1B3DZNNOQISOI,B000HWXS2S,425407477,With Oden on Our Side,Music,5,0,0,N,Y,Five Stars,Great album!,2015-07-22
196,US,584561,R1LSKDY6VVAGMX,0446310786,994527755,To Kill a Mockingbird,Books,5,1,1,N,N,A Classic That Still Rings True,I recently re-read TO KILL A MOCKINGBIRD afte...,2015-07-22
197,US,19267369,R1VVUISCINK9FI,B00D7Z4GQY,400840444,The Husband's Secret,Digital_Ebook_Purchase,5,0,0,N,Y,What would you do?,I kept thinking about the choices made by the ...,2015-07-22
198,US,367067,RBUD4J9GVPNDE,B00JUW71QK,393416784,SainSmart Ultimate Starter Kit RFID Master wit...,PC,5,0,1,N,Y,Five Stars,Everything I have tried so far works,2015-07-22


In [ ]:
import pandas as pd
import csv
import os

input_path = "E:/e-commerce-dataset/archive/amazon_reviews_multilingual_US_v1_00.tsv"
output_dir = "E:/ecommerce-data-lake/raw/amazon_reviews/category=multilingual"

chunksize = 200_000
bad_rows = 0
total_rows = 0

os.makedirs(output_dir, exist_ok=True)

# 🔥 handler đếm dòng lỗi
def bad_line_handler(line):
    global bad_rows
    bad_rows += 1
    return None   # skip dòng

reader = pd.read_csv(
    input_path,
    sep="\t",
    chunksize=chunksize,
    engine="python",
    quoting=csv.QUOTE_NONE,
    on_bad_lines=bad_line_handler
)

for i, chunk in enumerate(reader):
    total_rows += len(chunk)

    print(f"Chunk {i} processed | rows={len(chunk)} | total={total_rows}")

    chunk.to_parquet(
        os.path.join(output_dir, f"part_{i:03d}.parquet"),
        index=False,
        compression="snappy"
    )

print("\n===== SUMMARY =====")
print("Total valid rows:", total_rows)
print("Total skipped rows:", bad_rows)

Chunk 0 processed | rows=200000 | total=200000
Chunk 1 processed | rows=200000 | total=400000
Chunk 2 processed | rows=200000 | total=600000
Chunk 3 processed | rows=200000 | total=800000
Chunk 4 processed | rows=200000 | total=1000000
Chunk 5 processed | rows=200000 | total=1200000
Chunk 6 processed | rows=200000 | total=1400000
Chunk 7 processed | rows=200000 | total=1600000
Chunk 8 processed | rows=200000 | total=1800000
Chunk 9 processed | rows=200000 | total=2000000
Chunk 10 processed | rows=200000 | total=2200000
Chunk 11 processed | rows=200000 | total=2400000
Chunk 12 processed | rows=200000 | total=2600000
Chunk 13 processed | rows=200000 | total=2800000
Chunk 14 processed | rows=200000 | total=3000000
Chunk 15 processed | rows=200000 | total=3200000
Chunk 16 processed | rows=200000 | total=3400000
Chunk 17 processed | rows=200000 | total=3600000
Chunk 18 processed | rows=200000 | total=3800000
Chunk 19 processed | rows=200000 | total=4000000
Chunk 20 processed | rows=200000 |

In [ ]:
df1 = pd.read_parquet(f"{output_dir}/part_032.parquet")
df1.head(20)

,marketplace,customer_id,review_id,product_id,product_parent,product_title,product_category,star_rating,helpful_votes,total_votes,vine,verified_purchase,review_headline,review_body,review_date
0,US,17015584,R1PNLMKXXC9AL1,B0040I09RM,117036067,Sons of Anarchy Season 1,Digital_Video_Download,5,0,0,N,Y,Five Stars,AAAA,2015-05-30
1,US,12652834,R1110DNB8Y9A87,B00L9B7IKE,627793267,The Girl on the Train: A Novel,Digital_Ebook_Purchase,4,0,0,N,Y,Fun read,Very exciting book. Hard to put down. Charac...,2015-05-30
2,US,17015584,R14N5S8RQIW9GM,B003M70O5O,193295427,Sons of Anarchy Season 2,Digital_Video_Download,5,0,0,N,Y,Five Stars,AAA,2015-05-30
3,US,12541110,R2KIMGIPCN1AQQ,B0084AXZK0,118508203,The Picture of Dorian Gray,Digital_Ebook_Purchase,2,0,2,N,Y,Two Stars,Overrated,2015-05-30
4,US,16158971,RFSVSSDXOIFR8,B00HZ4CYOY,647510225,King of Math Junior,Mobile_Apps,4,1,1,N,Y,"Not worth it for pre readers, good practice ap...",I like the concept and the levels are nice. M...,2015-05-30
5,US,12485569,RWEX0QXLAM6E6,B004LQI0HE,646177839,Never Let Me Go,Digital_Video_Download,5,0,0,N,Y,Read the novel- beautifully written!,Stays with you long after it ends. Read the no...,2015-05-30
6,US,16158971,R166AD913AWNX6,B00SWWPVSY,855903037,Odd Bot Out,Mobile_Apps,5,0,0,N,Y,Nice unique puzzle app,There are 100 increasingly challenging mini le...,2015-05-30
7,US,12443445,R3EQX3ZU0FLNJ7,B004GJDQT8,36653526,Amazon Underground,Mobile_Apps,4,0,0,N,Y,Four Stars,You can find most anything,2015-05-30
8,US,16088396,R23TS6G6YDHXJR,B00JAC25FM,498232276,Farm Mania,Mobile_Apps,4,1,1,N,Y,addicting,good way to pass time and stimulate the mind w...,2015-05-30
9,US,12348168,R15CQQHQ3FDPUQ,B00QW8TYWO,828652896,Crossy Road,Mobile_Apps,4,0,0,N,N,Four Stars,Cool game. I like it.,2015-05-30


In [ ]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   marketplace        200000 non-null  object
 1   customer_id        200000 non-null  int64 
 2   review_id          200000 non-null  object
 3   product_id         200000 non-null  object
 4   product_parent     200000 non-null  int64 
 5   product_title      200000 non-null  object
 6   product_category   200000 non-null  object
 7   star_rating        200000 non-null  int64 
 8   helpful_votes      200000 non-null  int64 
 9   total_votes        200000 non-null  int64 
 10  vine               200000 non-null  object
 11  verified_purchase  200000 non-null  object
 12  review_headline    199990 non-null  object
 13  review_body        199979 non-null  object
 14  review_date        200000 non-null  object
dtypes: int64(5), object(10)
memory usage: 22.9+ MB


In [ ]:
import torch
from transformers import pipeline

# Kiểm tra thiết bị (0 là GPU, -1 là CPU)
device = 0 if torch.cuda.is_available() else -1
print(f"Chatbot sẽ chạy trên: {'GPU' if device == 0 else 'CPU'}")

# Chạy thử một model phân tích cảm xúc (Sentiment Analysis) đơn giản
nlp = pipeline("sentiment-analysis", device=device)
print(nlp("I love this product, it is amazing!"))


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Chatbot sẽ chạy trên: GPU


c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:00,

[{'label': 'POSITIVE', 'score': 0.9998867511749268}]


In [ ]:
import psycopg2

try:
    db_host = os.getenv("localhost")
    db_user = os.getenv("username")
    db_password = os.getenv("password")
    db_port = os.getenv("port")
    db_database = os.getenv("database")
    

    connection = psycopg2.connect(
        user=db_user,
        password=db_password,
        host=db_host,
        port=db_port,
        database=db_database
    )

    cursor = connection.cursor()
    # Kiểm tra thử kết nối
    cursor.execute("SELECT version();")
    record = cursor.fetchone()
    print("Bạn đang kết nối với:", record)

except (Exception, psycopg2.Error) as error:
    print("Lỗi khi kết nối với PostgreSQL:", error)

finally:
    if connection:
        cursor.close()
        connection.close()
        print("Đã đóng kết nối.")

Bạn đang kết nối với: ('PostgreSQL 16.10, compiled by Visual C++ build 1944, 64-bit',)
Đã đóng kết nối.
